# 02 — Embeddings & Storage

Converts parsed transcripts into embeddings and stores them in ChromaDB with metadata.

### Key concepts
- **Embedding**: A numerical vector that captures the semantic meaning of text. Similar meanings produce similar vectors.
- **Vector store**: A database that stores embeddings and supports similarity search. We use ChromaDB.
- **Metadata**: Structured data attached to each chunk (company, quarter, speaker, role) that enables filtering during retrieval.

In [ ]:
import json
from pathlib import Path
import numpy as np
import chromadb
from chromadb.utils import embedding_functions

## Load transcripts

Read all the JSON files produced by notebook 01. These contain the speaker turns with text, speaker name, and role — the raw material we need to create searchable embeddings.

In [ ]:
processed_dir = Path("../data/processed")
all_transcripts = []
for path in sorted(processed_dir.glob("*.json")):
    with open(path, "r", encoding="utf-8") as f:
        all_transcripts.append(json.load(f))
    print(f"  Loaded {path.name}")

total_turns = sum(t["total_turns"] for t in all_transcripts)
print(f"\n{len(all_transcripts)} transcripts, {total_turns} turns total")

## How embeddings work

An embedding model converts text into a fixed-size numerical vector where **similar meanings map to nearby points**. This is what enables searching by meaning rather than exact keywords. We use `all-MiniLM-L6-v2` — lightweight (runs on CPU) and produces 384-dimensional vectors.

In [ ]:
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# Semantic similarity demo
samples = [
    "Revenue grew 15% year over year",
    "Sales increased significantly compared to last year",
    "The weather in Tokyo is sunny today",
]
embs = embedding_fn(samples)

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(f"Embedding dimensions: {len(embs[0])}")
print(f"\n'Revenue grew...' vs 'Sales increased...':  {cosine_similarity(embs[0], embs[1]):.3f}  (similar meaning)")
print(f"'Revenue grew...' vs 'Weather in Tokyo...':   {cosine_similarity(embs[0], embs[2]):.3f}  (unrelated)")

## Build documents, metadata, and IDs

ChromaDB needs three parallel lists: `documents` (the text), `metadatas` (structured fields for filtering), and `ids` (unique identifiers). Each speaker turn becomes one document. The metadata carries company, quarter, speaker, and role — this is what allows filtering like "only CFO statements in Q4-2025" at query time.

In [ ]:
documents = []
metadatas = []
ids = []

for transcript in all_transcripts:
    for i, turn in enumerate(transcript["turns"]):
        documents.append(turn["text"])
        metadatas.append({
            "company": transcript["company"],
            "quarter": transcript["quarter"],
            "speaker": turn["speaker"],
            "role": turn["role"],
        })
        ids.append(f"{transcript['company']}_{transcript['quarter']}_{i}")

print(f"Documents: {len(documents)} | Unique IDs: {len(set(ids))}")
print(f"Sample ID: {ids[0]}")
print(f"Sample metadata: {metadatas[0]}")

## Store in ChromaDB

ChromaDB automatically generates embeddings when we `add()` documents — we just pass the text and it uses our `embedding_fn` internally. Data is persisted to disk so it survives between sessions. We delete and recreate the collection to ensure a clean state on re-runs.

In [ ]:
client = chromadb.PersistentClient(path="../chroma_db")

try:
    client.delete_collection("earnings_calls")
except Exception:
    pass

collection = client.create_collection(
    name="earnings_calls",
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"},
)

BATCH_SIZE = 50
for i in range(0, len(documents), BATCH_SIZE):
    end = min(i + BATCH_SIZE, len(documents))
    collection.add(
        documents=documents[i:end],
        metadatas=metadatas[i:end],
        ids=ids[i:end],
    )

print(f"Stored {collection.count()} documents in ChromaDB")

## Verify with a quick query

Confirm the data is stored and queryable. Results should come from different quarters and show relevant speakers — if everything comes from one quarter or irrelevant speakers, something went wrong in the metadata.

In [ ]:
results = collection.query(query_texts=["What were the revenue results?"], n_results=3)

for i in range(len(results["documents"][0])):
    meta = results["metadatas"][0][i]
    dist = results["distances"][0][i]
    print(f"[{meta['quarter']}] {meta['speaker']} (dist: {dist:.3f})")
    print(f"  {results['documents'][0][i][:120]}...\n")

In [ ]:
# Collection summary
all_docs = collection.get(include=["metadatas"])
quarters = {}
for m in all_docs["metadatas"]:
    q = m["quarter"]
    quarters[q] = quarters.get(q, 0) + 1

print("Collection summary:")
for q in sorted(quarters):
    print(f"  {q}: {quarters[q]} turns")